In [0]:
%sql
SELECT * from hive_streamming.silver.test ORDER BY viewer_id,snapshot_index

In [0]:
%sql
select start_buffering_ts,end_buffering_ts,case when start_buffering_ts is not null and end_buffering_ts is null then 

In [0]:
%sql
UPDATE hive_streamming.silver.gold_watermark
SET last_processed_ts = TIMESTAMP('2025-12-13T16:21:15.000+00:00')
WHERE table_name = 'test';


In [0]:
DESCRIBE HISTORY hive_streamming.silver.test

In [0]:
%sql
ALTER TABLE hive_streamming.silver.test
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
dbutils.fs.rm(
  "abfss://bronze@bhupeshstorage001.dfs.core.windows.net/test/",
  recurse=True
)

In [0]:
dbutils.fs.ls("abfss://bronze@bhupeshstorage001.dfs.core.windows.net/")

In [0]:
%sql
SELECT
  viewer_id,
  filedate,
  avg_bitrate_mbps,
  total_buffering_sec,

  CASE
    WHEN avg_bitrate_mbps < 2
         AND total_buffering_sec > 0
      THEN 'RECOVERY_PHASE (Low bitrate + Buffering)'

    WHEN avg_bitrate_mbps >= 2
         AND total_buffering_sec > 0
      THEN 'OVERSHOOT (High bitrate + Buffering)'

    WHEN avg_bitrate_mbps < 2
         AND total_buffering_sec = 0
      THEN 'STABLE_LOW_QUALITY (Low bitrate + No buffering)'

    WHEN avg_bitrate_mbps >= 2
         AND total_buffering_sec = 0
      THEN 'IDEAL_QOE (High bitrate + No bufferings)'

    ELSE 'UNKNOWN'
  END AS qoe_state

FROM hive_streamming.gold.fact_qoe_session;


In [0]:
%sql
SELECT
  viewer_id,
  filedate,
  avg_bitrate_mbps,
  total_buffering_sec,
  quality_switch_count,

  CASE
    WHEN avg_bitrate_mbps < 2
         AND total_buffering_sec > 10
      THEN 'LOW_BITRATE_WITH_BUFFERING'

    WHEN avg_bitrate_mbps < 2
         AND total_buffering_sec <= 10
      THEN 'LOW_BITRATE_NO_BUFFERING'

    WHEN avg_bitrate_mbps >= 2
         AND total_buffering_sec > 10
      THEN 'BUFFERING_WITH_GOOD_BITRATE'

    WHEN quality_switch_count > 3
      THEN 'UNSTABLE_NETWORK'

    ELSE 'GOOD_QOE'
  END AS qoe_root_cause

FROM hive_streamming.gold.fact_qoe_session;


In [0]:
 SELECT
  CASE
    WHEN avg_bitrate_mbps >= 3 THEN 'HIGH (>=3 Mbps)'
    WHEN avg_bitrate_mbps >= 2 THEN 'MEDIUM (2–3 Mbps)'
    ELSE 'LOW (<2 Mbps)'
  END AS bitrate_bucket,
  AVG(total_buffering_sec) AS avg_buffering_sec,
  COUNT(*) AS sessions
FROM hive_streamming.gold.fact_qoe_session
GROUP BY bitrate_bucket
ORDER BY bitrate_bucket;


In [0]:
  SELECT
  CASE
    WHEN total_buffering_sec = 0 THEN 'NO BUFFERING'
    WHEN total_buffering_sec <= 10 THEN 'LOW BUFFERING'
    ELSE 'HIGH BUFFERING'
  END AS buffering_bucket,
  AVG(avg_bitrate_mbps) AS avg_bitrate,
  COUNT(*) AS sessions
FROM hive_streamming.gold.fact_qoe_session
GROUP BY buffering_bucket
ORDER BY buffering_bucket;


In [0]:
%sql
SELECT
  CASE
    WHEN avg_bitrate_mbps >= 3 THEN 'High Bitrate'
    WHEN avg_bitrate_mbps >= 2 THEN 'Medium Bitrate'
    ELSE 'Low Bitrate'
  END AS bitrate_bucket,

  CASE
    WHEN total_buffering_sec > 10 THEN 'UNSTABLE'
    ELSE 'STABLE'
  END AS stability,

  ROUND(AVG(total_buffering_sec), 2) AS avg_buffering_sec,
  COUNT(*) AS sessions

FROM hive_streamming.gold.fact_qoe_session
GROUP BY bitrate_bucket, stability
ORDER BY bitrate_bucket, stability;


In [0]:
%sql
select * from hive_streamming.gold.fact_buffering_event

In [0]:
%sql
select * from hive_streamming.gold.fact_qoe_session